## 1:Import required libraries

In [24]:
import random
import pandas as pd
from faker import Faker
from datetime import datetime

### 2:Database creation and connection 

In [25]:
import sqlite3
conn = sqlite3.connect("retail.db")
cursor=conn.cursor()
print("Database Connected")

Database Connected


## 3:Table Creation

In [26]:
customers = pd.read_csv("customers.csv")
products = pd.read_csv("cleaned_products.csv")
orders = pd.read_csv("cleaned_orders.csv")
order_items = pd.read_csv("order_items.csv")

In [31]:
customers.to_sql("customers",conn,if_exists="replace",index=False)

products.to_sql("products",conn,if_exists="replace",index=False)

orders.to_sql("orders",conn,if_exists="replace",index=False)

order_items.to_sql("order_items",conn,if_exists="replace",index=False)

500

## 4:Query run function

In [28]:
def run_query(query):
    return pd.read_sql_query(query,conn)

#### 7:Calculate running total of revenue per region, ordered by date.
#### Show: region_code, order_date, daily_revenue, running_total


In [33]:

q7 = """
with daily_sales as (
    select o.regions,DATE(o.order_date) as order_date,sum(
            oi.quantity * oi.unit_price * (1 - oi.discount_price / 100.0)) AS daily_revenue from orders o
join order_items oi
on o.order_id = oi.order_id
group by o.regions, date(o.order_date))
select regions,order_date,daily_revenue,
sum(daily_revenue) over (partition by regions order by order_date) as running_total
from daily_sales
order by regions,order_date;
"""
df_q7=run_query(q7)
df_q7

,regions,order_date,daily_revenue,running_total
0,EAST,2024-07-24,8496.44,8496.44
1,EAST,2024-07-25,10034.50,18530.94
2,EAST,2024-07-29,2929.11,21460.05
3,EAST,2024-08-11,14651.80,36111.85
4,EAST,2024-08-19,18831.90,54943.75
...,...,...,...,...
288,WEST,2026-05-27,9635.82,631426.08
289,WEST,2026-05-31,16638.78,648064.86
290,WEST,2026-06-07,13004.94,661069.80
291,WEST,2026-07-02,5313.60,666383.40


##### 8:For each category, rank products by total revenue.

##### Show: category, product_name, total_revenue, rank_in_category

#### Products with same revenue should have same rank.


In [11]:

q8="""select category,product_name,total_revenue,
dense_rank() over(partition by category order by total_revenue desc
) as rank_in_category
from( select p.category,p.product_name,
sum( oi.quantity *oi.unit_price *(1 - oi.discount_price/100.0)) as total_revenue
from products p
join order_items oi
on p.product_id=oi.product_id
group by p.category,p.product_name)
"""
df_q8=run_query(q8)
df_q8      

,category,product_name,total_revenue,rank_in_category
0,Books,Cpp,143944.61,1
1,Books,Alchemist,141290.85,2
2,Books,Omg,127328.11,3
3,Books,Mice,110552.84,4
4,Books,Heaven,101811.95,5
5,Books,Tillu,82678.36,6
6,Clothing,Lower,227837.93,1
7,Clothing,Pajama,177602.49,2
8,Clothing,Jeans,143152.12,3
9,Clothing,Skirt,76553.64,4


##### 9:For each customer, calculate days between consecutive orders.
##### Show: customer_id, order_date, previous_order_date, days_gap
##### Flag customers with average gap > 30 days as "At Risk"

In [13]:
q9="""with cte as (
select customer_id,order_date,
lag(order_date) over (partition by customer_id order by order_date) as previous_order_date
from orders)

select customer_id,order_date,previous_order_date,
round(julianday(order_date) - julianday(previous_order_date)) as days_gap
from cte
"""
df_q9=run_query(q9)
df_q9.tail()

,customer_id,order_date,previous_order_date,days_gap
495,unknown,2026-01-27 14:37:39,2026-01-23 17:25:51,4.0
496,unknown,2026-02-11 03:52:20,2026-01-27 14:37:39,15.0
497,unknown,2026-04-12 11:16:21,2026-02-11 03:52:20,60.0
498,unknown,2026-06-14 02:27:14,2026-04-12 11:16:21,63.0
499,unknown,2026-06-21 20:17:44,2026-06-14 02:27:14,8.0


##### 10:Using CTEs, find:
##### - First, calculate monthly revenue per customer
##### - Then, categorize customers: 'High' (>10000), 'Medium' (5000-10000), 'Low' (<5000)
##### - Finally, show count of customers in each category per month


In [15]:
q10="""with monthly_revenue as(
select o.customer_id,
strftime('%Y-%m',o.order_date) AS month,
sum(oi.quantity*oi.unit_price*(1-oi.discount_price/100.0)) revenue
from orders o
join order_items oi
on o.order_id=oi.order_id
group by o.customer_id,month),

customer_type as(
select *,
case when revenue>10000 then 'High'
     when revenue>=5000 then 'Medium'
     else 'Low'
end as category
from monthly_revenue)

select month,category,count(customer_id) customers
from customer_type
group by month,category
order by month
"""

df_q10=run_query(q10)
df_q10

,month,category,customers
0,2024-07,High,4
1,2024-07,Low,5
2,2024-07,Medium,4
3,2024-08,High,6
4,2024-08,Low,6
...,...,...,...
66,2026-06,High,3
67,2026-06,Low,6
68,2026-06,Medium,1
69,2026-07,High,2


#### 12:Compare each month's revenue with same month previous year.
##### Show: year, month, revenue, prev_year_revenue, yoy_growth_percent
##### Handle cases where previous year data doesn't exist.

In [16]:
q12="""with monthly_revenue as(
select strftime('%Y',order_date) year, strftime('%m',order_date) month,
sum(oi.quantity*oi.unit_price*(1-oi.discount_price/100.0)) revenue
from orders o
join order_items oi
on o.order_id=oi.order_id
group by year,month
)

select year,month,revenue,
lag(revenue) over(
partition by month
order by year) prev_year_revenue,
round(
(revenue-lag(revenue) over(partition by month order by year))*100.0/

lag(revenue) over(
partition by month order by year),2
) yoy_growth_percent
from monthly_revenue;
"""
df_q12=run_query(q12)
df_q12

,year,month,revenue,prev_year_revenue,yoy_growth_percent
0,2025,01,203309.23,NaN,NaN
1,2026,01,173060.36,203309.23,-14.88
2,2025,02,24731.96,NaN,NaN
3,2026,02,101338.05,24731.96,309.75
4,2025,03,92350.37,NaN,NaN
5,2026,03,137089.39,92350.37,48.44
6,2025,04,48603.39,NaN,NaN
7,2026,04,157584.98,48603.39,224.23
8,2025,05,95866.72,NaN,NaN
9,2026,05,159681.43,95866.72,66.57


##### 13:For each customer, show their first purchased category and most recent purchased category.
##### Flag if they are different (category_shift = 'Yes'/'No')

In [17]:
q13="""with cte as (
select o.customer_id,p.category,
row_number() over(partition by o.customer_id order by o.order_date) rn1,
row_number() over(partition by o.customer_id order by o.order_date desc) rn2
from orders o
join order_items oi on o.order_id = oi.order_id
join products p on oi.product_id = p.product_id
)

select a.customer_id,a.category as first_category,b.category as last_category,
case when a.category = b.category then 'No'
     else 'Yes'
end as category_shift
from cte a
join cte b
on a.customer_id = b.customer_id
and a.rn1 = 1
and b.rn2 = 1;
"""
df_q13=run_query(q13)
df_q13

,customer_id,first_category,last_category,category_shift
0,1.0,Books,Electronics,Yes
1,10.0,Books,Clothing,Yes
2,101.0,Home,Home,No
3,102.0,Clothing,Clothing,No
4,103.0,Home,Home,No
...,...,...,...,...
223,9.0,Electronics,Electronics,No
224,91.0,Electronics,Electronics,No
225,96.0,Clothing,Home,Yes
226,98.0,Clothing,Clothing,No


##### 14:Calculate what percentage of total revenue comes from top N% of customers.
##### Show: customer_id, revenue, cumulative_revenue, cumulative_percent

In [19]:
q14="""with customer_revenue as(
select o.customer_id,
sum(oi.quantity * oi.unit_price *(1 - oi.discount_price/100.0)) as revenue
from orders o
join order_items oi
on o.order_id = oi.order_id
group by o.customer_id
)

select customer_id,revenue,
sum(revenue) over(order by revenue desc) as cumulative_revenue,
round(sum(revenue) over(
order by revenue desc) * 100.0 /sum(revenue) over(),2) as cumulative_percent
from customer_revenue
order gy revenue desc
"""
df_q14=run_query(q14)
df_q14

,customer_id,revenue,cumulative_revenue,cumulative_percent
0,unknown,253885.13,253885.13,8.47
1,343.0,73498.55,327383.68,10.92
2,245.0,71957.44,399341.12,13.32
3,391.0,58512.32,457853.44,15.27
4,37.0,48443.28,506296.72,16.89
...,...,...,...,...
223,426.0,-60.78,3020622.29,100.77
224,486.0,-656.76,3019965.53,100.75
225,159.0,-1859.56,3018105.97,100.69
226,86.0,-6139.30,3011966.67,100.48


##### 16:Find products frequently bought together.
##### Show: product_a, product_b, times_bought_together
##### Exclude same product pairs and duplicates (A-B and B-A should appear once)

In [22]:
q16="""select oi1.product_id as product_a,
oi2.product_id as product_b,
count(*) as times_bought_together
from order_items oi1
join order_items oi2
on oi1.order_id = oi2.order_id
and oi1.product_id < oi2.product_id
group by oi1.product_id, oi2.product_id
"""
df_q16=run_query(q16)
df_q16

,product_a,product_b,times_bought_together
0,4,487,1
1,5,134,1
2,12,316,1
3,12,328,1
4,13,225,1
...,...,...,...
245,435,447,1
246,435,470,1
247,447,470,1
248,450,458,1
